# 03 · From-Scratch Two-View Reconstruction  ← **Milestone Artifact #1**

**Goal:** implement the normalized 8-point algorithm + essential-matrix
decomposition + DLT triangulation + cheirality check **from scratch**, then
compare the result against cv2's reference (`findEssentialMat` +
`recoverPose` + `triangulatePoints`).

All from-scratch code lives in `src/two_view.py` between
`### YOUR CODE HERE` markers. This notebook only orchestrates the calls.

### What "from scratch" means here

- No `cv2.findFundamentalMat`, `cv2.findEssentialMat`, `cv2.recoverPose`,
  or `cv2.triangulatePoints` in the from-scratch path.
- `np.linalg.svd`, `np.linalg.norm`, matrix multiplications: fine
  (linear algebra, not vision).
- cv2 is allowed only in the reference path used for comparison.

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import two_view, viz
import cv2

TREE_ID = "tree_oak_01"
IDX_A = 0
# Notebook 02 picks IDX_B = len(frame_paths) // 4 dynamically; auto-discover the cache.
recon_dir = PROJECT_ROOT / "outputs" / "reconstructions"
candidates = sorted(recon_dir.glob(f"{TREE_ID}_pair_{IDX_A}_*.npz"))
if not candidates:
    raise FileNotFoundError(
        f"No cached correspondence pair {TREE_ID}_pair_{IDX_A}_*.npz in {recon_dir}.\n"
        "Run notebook 02 first to detect/verify SIFT matches and cache them."
    )
CACHE = candidates[-1]
IDX_B = int(CACHE.stem.rsplit("_", 1)[-1])
print(f"Using cached pair: {CACHE.name}  (IDX_A={IDX_A}, IDX_B={IDX_B})")

data = np.load(CACHE)
pts1, pts2, K = data["pts1"], data["pts2"], data["K"]
print(f"{len(pts1)} verified correspondences loaded.")


## 1. Run the from-scratch pipeline

`two_view.two_view_reconstruction` chains:

1. Hartley normalisation
2. Normalised 8-point → F
3. F → E (and re-project onto the essential manifold)
4. E → 4 candidate (R, t) hypotheses
5. DLT triangulation + cheirality vote → winning (R, t, X)

In [ ]:
result = two_view.two_view_reconstruction(pts1, pts2, K)
print("From-scratch R =\n", result.R)
print("From-scratch t (unit) =", result.t.ravel())
print(f"{int(result.inlier_mask.sum())}/{len(pts1)} points pass cheirality.")

## 2. cv2 reference for comparison

We are NOT using cv2 for our reconstruction — we're using it to verify ours.
Expect a near-identical R (rotation MAE under ~0.5°) and a t that agrees up to
sign (translation scale is unrecoverable from two views).

In [ ]:
E_cv2, _ = cv2.findEssentialMat(pts1, pts2, K, method=cv2.RANSAC, threshold=1.0)
n_inl, R_cv2, t_cv2, _ = cv2.recoverPose(E_cv2, pts1, pts2, K)
print("cv2 R =\n", R_cv2)
print("cv2 t (unit) =", t_cv2.ravel())

# Rotation difference in degrees (geodesic on SO(3)).
R_diff = result.R @ R_cv2.T
cos_angle = (np.trace(R_diff) - 1) / 2
rot_err_deg = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
# Translation: align signs first because cv2 may pick opposite direction.
t_ours = result.t.ravel() / (np.linalg.norm(result.t) + 1e-12)
t_cv = t_cv2.ravel() / (np.linalg.norm(t_cv2) + 1e-12)
if np.dot(t_ours, t_cv) < 0:
    t_cv = -t_cv
t_err_deg = np.degrees(np.arccos(np.clip(np.dot(t_ours, t_cv), -1.0, 1.0)))
print(f"\nRotation error (geodesic): {rot_err_deg:.3f}°")
print(f"Translation direction error:  {t_err_deg:.3f}°")

## 3. Side-by-side visualisation — saved as the milestone figure

In [ ]:
# cv2 triangulation under cv2's (R, t) — gives the reference 3D cloud.
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
P2_cv = K @ np.hstack([R_cv2, t_cv2])
X_cv2_h = cv2.triangulatePoints(P1, P2_cv, pts1.T, pts2.T)
X_cv2 = (X_cv2_h[:3] / X_cv2_h[3]).T

# Take only cheirality-positive points on each side.
X_ours = result.points_3d[result.inlier_mask]
X_cv2_in = X_cv2[X_cv2[:, 2] > 0]

from matplotlib.figure import Figure
fig = plt.figure(figsize=(14, 7))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax1.scatter(X_ours[:, 0], X_ours[:, 1], X_ours[:, 2], s=4, c="tab:green")
ax1.set_title(f"From-scratch (N={len(X_ours)})")
ax2.scatter(X_cv2_in[:, 0], X_cv2_in[:, 1], X_cv2_in[:, 2], s=4, c="tab:blue")
ax2.set_title(f"cv2 reference (N={len(X_cv2_in)})")
for ax, pts in [(ax1, X_ours), (ax2, X_cv2_in)]:
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
fig.suptitle("Two-view reconstruction — from-scratch vs. OpenCV reference")
viz.save_fig(fig, "milestone_two_view.png")
plt.show()

## 4. Sampson diagnostic

A second sanity check: feed the from-scratch F (or E·K) into the Sampson
distance and confirm most inliers have sub-pixel² error.

In [ ]:
if result.F is not None:
    d = two_view.sampson_distance(result.F, pts1, pts2)
    print(f"Sampson distance — median {np.median(d):.3f} px², 95th {np.percentile(d, 95):.3f} px²")
else:
    # We always have E. Convert back to F for diagnostics.
    F_from_E = np.linalg.inv(K).T @ result.E @ np.linalg.inv(K)
    d = two_view.sampson_distance(F_from_E, pts1, pts2)
    print(f"Sampson distance (via E→F) — median {np.median(d):.3f} px², 95th {np.percentile(d, 95):.3f} px²")